# NYC Bike Data: Dataset Description & Exploratory Analysis

This notebook documents and explores every dataset used in the project and underpins the
**technical report** in `docs/technical-report/`. It covers, in order of importance:

1. **Citi Bike dataset**: the project's main data source with bike trip records (one monthly file, downloaded manually).
2. **Station metadata**: current Citi Bike stations from the Lyft GBFS feed (fetched live).
3. **Weather**: hourly NYC observations from the Open-Meteo archive (fetched live).
4. **Bike lanes**: NYC bike-route network from NYC OpenData (fetched live).

For each dataset we describe its **structure**, run **exploratory data analysis (EDA)**, and
assess **data quality**. See `src/notebooks/README.md` for how to obtain the ride data and run
this notebook.

## 0. Setup

In [ ]:
import json
import re
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
import folium
from folium.plugins import HeatMap

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Where to find the manually-downloaded Citi Bike monthly CSV(s) (relative to this notebook).
DATA_DIR = Path("data")

# Data sources (mirrors src/ingestion/config.yaml and src/backend/config.yaml).
WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/archive"
BIKE_ROUTES_URL = "https://data.cityofnewyork.us/api/views/mzxg-pwib/rows.csv?accessType=DOWNLOAD"
GBFS_INFO_URL = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json"
GBFS_STATUS_URL = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_status.json"
NYC_LAT, NYC_LON = 40.7823234, -73.9654161
NYC_TZ = "America/New_York"

# Distance-feature constants (see src/ingestion/sources/distances.py).
EARTH_RADIUS_KM = 6371
STREET_CIRCUITY_FACTOR = 1.3

Here we define an utility function for computing the Haversine distance calculated as:

$$d = 2 r \arcsin \sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1) \cos(\phi_2) \sin^2\left(\frac{\Delta \lambda}{2}\right)}$$

where $\phi$ is latitude, $\lambda$ is longitude, and $r$ is the Earth's radius (6371 km).

In [ ]:
# Vectorised great-circle distance (km) scaled by the street-circuity factor,
# matching the production formula in src/ingestion/sources/distances.py.
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_KM * c * STREET_CIRCUITY_FACTOR

## 1. Citi Bike Trip Data

The core dataset Citi Bike publishes one CSV per month, each row a single trip, available from
the [system data page](https://www.citibikenyc.com/system-data). Download one month and place the
extracted CSV in the `data/` folder next to this notebook (see `README.md`). The analysis shown here uses the **February 2025** file (`202502-citibike-tripdata`).

Newer files (2020+) and the legacy (pre-2020) schema use different column names. This map normalises the legacy columns to the modern schema so the rest of the notebook works regardless of which month was downloaded.

In [ ]:
LEGACY_RENAME = {
    "starttime": "started_at",
    "stoptime": "ended_at",
    "start station name": "start_station_name",
    "start station id": "start_station_id",
    "end station name": "end_station_name",
    "end station id": "end_station_id",
    "start station latitude": "start_lat",
    "start station longitude": "start_lng",
    "end station latitude": "end_lat",
    "end station longitude": "end_lng",
}

def load_rides(data_dir: Path):
    csv_files = sorted(data_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found in {data_dir.resolve()}. Download a Citi Bike monthly "
            "trip file, extract it, and place the CSV here. See src/notebooks/README.md."
        )
    df = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)

    # Columns unique to the legacy schema — kept track of for the data-quality section.
    legacy_cols = [c for c in ["gender", "birth year", "bikeid", "tripduration", "usertype"]
                   if c in df.columns]
    is_legacy = "rideable_type" not in df.columns
    if is_legacy:
        df = df.rename(columns=LEGACY_RENAME)
        if "usertype" in df.columns:
            df["member_casual"] = df["usertype"].map(
                {"Subscriber": "member", "Customer": "casual"}
            ).fillna(df["usertype"])
        df["rideable_type"] = pd.NA          # absent before the e-bike rollout
        if "ride_id" not in df.columns:
            df["ride_id"] = df.index.astype(str)

    # Recent months carry fractional seconds meanwhile older months do not. Use a mixed format to handle both.
    for col in ["started_at", "ended_at"]:
        try:
            df[col] = pd.to_datetime(df[col], format="ISO8601")
        except (ValueError, TypeError):
            df[col] = pd.to_datetime(df[col], format="mixed")
    return df, {"files": [f.name for f in csv_files], "is_legacy": is_legacy,
                "legacy_cols": legacy_cols}

df, meta = load_rides(DATA_DIR)
print("Loaded files :", meta["files"])
print("Schema       :", "legacy (pre-2020)" if meta["is_legacy"] else "modern (2020+)")
print("Shape        :", df.shape)
df.head()

### 1.1 Structure

We provide a brief overview of the dataset's structure, including column names, data types, and sample records, through the `pandas` library. 

In [ ]:
df.info()

print("\nUnique rideable_type :", df["rideable_type"].dropna().unique().tolist())
print("Unique member_casual :", df["member_casual"].dropna().unique().tolist())

unique_stations = pd.concat([df["start_station_name"], df["end_station_name"]]).nunique()
print(f"Unique stations      : {unique_stations}")
print(f"Date range           : {df['started_at'].min()} -> {df['started_at'].max()}")

### 1.2 Trip Duration

Given the raw trip data, we compute a set of derived features, including trip duration and distance. This is done in order to enhance the analysis of user behaviour and trip patterns. 

In [ ]:
df["ride_length_s"] = (df["ended_at"] - df["started_at"]).dt.total_seconds()
df["ride_length_min"] = df["ride_length_s"] / 60

print("Ride length (minutes) summary:")
print((df["ride_length_s"] / 60).agg(["mean", "median", "min", "max", "std"]).round(2))

negative = (df["ride_length_s"] < 0).sum()
long_rides = (df["ride_length_s"] > 4 * 3600).sum()
print(f"\nRides with negative duration : {negative}")
print(f"Rides longer than 4 hours    : {long_rides}")

# Every trip is kept
plt.figure(figsize=(8, 5))
sns.histplot(df["ride_length_min"].clip(lower=0, upper=60), bins=60)
plt.axvline(df["ride_length_min"].median(), color="red", ls="--",
            label=f"median {df['ride_length_min'].median():.1f} min")
plt.title("Ride length (0\u201360 min)")
plt.xlabel("Ride length (min)")
plt.legend()
plt.tight_layout()
plt.show()

Ride length is strongly **right-skewed**: most trips last only a few to ~20 minutes (the
histogram is zoomed to the first hour), with a thin tail stretching to much longer rides. No
trips are excluded here. The few **negative** durations (clock/data glitches where a trip ends
before it starts) and rides over **4 hours** (more likely a bike that was never docked correctly
than a genuine 4-hour trip) are kept and folded into the edge bins, and are quantified above. The
distribution starts at 1 minute because Citi Bike removes trips shorter than 60 seconds at the
source (likely false starts or re-docking), so there are no zero-length trips.

### 1.3 Trip Distance

In [ ]:
# Straight-line distance between start and end stations, scaled by the circuity factor
df["trip_distance_km"] = haversine_km(
    df["start_lat"], df["start_lng"], df["end_lat"], df["end_lng"]
)

print("Trip distance (km) summary:")
print(df["trip_distance_km"].agg(["mean", "median", "min", "max", "std"]))

# Nothing is excluded
cap = 10  # km; distances above this collapse into the last bin
plt.figure(figsize=(8, 5))
sns.histplot(df["trip_distance_km"].clip(upper=cap), bins=60)
plt.xlabel("Trip distance (km)")
plt.title("Trip distance (all trips; tail >= 10 km folded)")
plt.tight_layout()
plt.show()

This is the **straight-line** distance between the start and end stations, scaled by a 1.3
circuity factor to approximate the real on-street path. No trips are excluded: **round trips**
that begin and end at the same station appear as the **0 km** bar (the straight-line metric
cannot capture their real path), and the long-distance tail is folded into the final bin
so these anomalies stay visible. Both must be considered when interpreting the distribution or
computing distance-based statistics (e.g., average distance per trip, average speed).

### 1.4 Bike Type and User Type

The objective of this analysis is to understand the relationship between bike type and user type. The first panel shows the **bike-type mix within each user type**. Meanwhile the second panel displays the **median ride length by user and bike type**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bike-type mix *within* each user type (share, not raw volume)
counts = df.groupby(["member_casual", "rideable_type"]).size().reset_index(name="rides")
counts["share"] = 100 * counts["rides"] / counts.groupby("member_casual")["rides"].transform("sum")
sns.barplot(data=counts, x="member_casual", y="share", hue="rideable_type", ax=axes[0])
axes[0].set(title="Bike-type mix by user type", xlabel="User type",
            ylabel="Share of the user's rides (%)")

# Median ride length by user and bike type
med = (df.groupby(["member_casual", "rideable_type"])["ride_length_min"]
         .median().reset_index())
sns.barplot(data=med, x="member_casual", y="ride_length_min", hue="rideable_type", ax=axes[1])
axes[1].set(title="Median ride length by user & bike type", xlabel="User type",
            ylabel="Median ride length (min)")

# Add headroom and pin each legend to the top-right so it never overlaps a bar.
for ax in axes:
    ax.set_ylim(0, ax.get_ylim()[1] * 1.3)
    ax.legend(title="rideable_type", loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()

# Overall composition, for context.
print("Share of rides by user type (%):")
print((df["member_casual"].value_counts(normalize=True) * 100).round(1))
print("\nShare of rides by bike type (%):")
print((df["rideable_type"].value_counts(normalize=True) * 100).round(1))

Raw volume is dominated by **members** (commuter use), but that count alone is not very
informative. Splitting each dimension by the other is more revealing. **Bike-type mix** (left):
**casual** riders lean more heavily on **electric** bikes than members do. **Ride length**
(right): casual riders take **longer** trips than members on both bike types, while members ride a
steady, short duration regardless of bike type, consistent with frequent short-hop commuting versus
occasional, mostly-electric leisure trips.

### 1.5 Temporal Patterns

Our purpose here is to explore the temporal patterns of bike usage, including daily and weekly trends, through the ride counts. 

In [ ]:
# Hour and weekday are taken from the END timestamp, matching the ingestion pipeline
df["hour"] = df["ended_at"].dt.hour
df["day_of_week"] = df["ended_at"].dt.day_name()
df["date"] = df["ended_at"].dt.date
df["day_type"] = np.where(
    df["ended_at"].dt.weekday < 5, "Weekday", "Weekend"
)

# Mean rides per hour, separating weekdays from weekends.
hourly = (
    df.groupby(["date", "hour", "day_type"]).size().reset_index(name="rides")
)
mean_rides = hourly.groupby(["hour", "day_type"])["rides"].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(x="hour", y="rides", hue="day_type", data=mean_rides)
plt.xlabel("Hour of day")
plt.ylabel("Mean rides per day")
plt.title("Mean rides by hour: weekday vs weekend")
plt.tight_layout()
plt.show()

Weekdays show the classic bimodal commuting pattern (peaks around 8 AM and 5-6 PM), while
weekends follow a single midday hump consistent with leisure use.

### 1.6 Spatial Patterns

We explore the spatial patterns of bike usage, using a spatial heatmap to visualize the distribution of bike trips across the city. This analysis helps identify areas with high demand for bike-sharing services.

In [ ]:
# Heatmap of trip origins, weighted by how many trips start at each station.
origins = (
    df.dropna(subset=["start_lat", "start_lng", "start_station_name"])
    .groupby("start_station_name")
    .agg(lat=("start_lat", "first"), lon=("start_lng", "first"), rides=("ride_id", "count"))
    .reset_index()
)
bike_map = folium.Map(location=[origins["lat"].mean(), origins["lon"].mean()], zoom_start=12)
HeatMap(origins[["lat", "lon", "rides"]].values.tolist()).add_to(bike_map)
# recreate the map and add a tighter heatmap to reduce the glow
bike_map = folium.Map(location=[origins["lat"].mean(), origins["lon"].mean()], zoom_start=12)
HeatMap(
    origins[["lat", "lon", "rides"]].values.tolist(),
    radius=8,
    blur=6,
    min_opacity=0.3,
    max_zoom=18
).add_to(bike_map)
bike_map

In [ ]:
# Station-to-station flow: how many trips run between each ordered station pair.
flow = (
    df.dropna(subset=["start_station_name", "end_station_name"])
    .groupby(["start_station_name", "end_station_name"]).size()
    .reset_index(name="trips")
)
print(f"Distinct station pairs with at least one trip: {len(flow):,}")
print("\nTop 10 station-to-station flows:")
print(flow.sort_values("trips", ascending=False).head(10).to_string(index=False))

Demand is highly concentrated: a small set of stations near transit hubs and the business
districts dominate both departures and arrivals, so the heatmap clusters around Midtown and Lower
Manhattan, and the busiest station-to-station flows tend to link these same hubs.

### 1.7 Data Quality

#### Missing Values

A small share of trips have missing values, and they are confined to the **station** columns.
The **end-of-trip** fields (`end_station_name`, `end_station_id`, `end_lat`, `end_lng`) are the most
affected probably due to trips ending away from a station. The non-station columns (`ride_id`, `rideable_type`, `started_at`,
`ended_at`, `member_casual`) have no missing values. The table below reports the missing count and
percentage for every raw column.

In [ ]:
raw_cols = ["ride_id", "rideable_type", "started_at", "ended_at",
            "start_station_name", "start_station_id", "end_station_name", "end_station_id",
            "start_lat", "start_lng", "end_lat", "end_lng", "member_casual"]
missing = df[raw_cols].isna().sum()
missing = (pd.DataFrame({"missing": missing, "pct (%)": (missing / len(df) * 100).round(2)})
           .sort_values("missing", ascending=False))
print(f"{len(df):,} trips loaded; missing values per column:")
missing

#### Trip Outliers

A handful of trips are clearly not ordinary rides. Some are **too long**
which points to a bike that was never returned rather than a continuous ride. Others cover **zero
straight-line distance** because they start and end at the same station (round trips the
point-to-point distance cannot capture). A few are **impossibly far** caused by
broken station coordinates (an endpoint at latitude/longitude 0, 0). The table below shows
examples of each.

In [ ]:
cols = ["started_at", "ride_length_min", "trip_distance_km",
        "start_station_name", "end_station_name", "member_casual"]
too_long = (df[(df["trip_distance_km"] > 0) & df["end_station_name"].notna()]
            .nlargest(3, "ride_length_min")[cols].assign(issue="too long"))
zero_dist = (df[df["trip_distance_km"] == 0]
             .nlargest(3, "ride_length_min")[cols].assign(issue="zero distance (round trip)"))
too_far = (df[df["trip_distance_km"] > 4000]
           .nlargest(2, "trip_distance_km")[cols].assign(issue="too far (bad coordinates)"))
print(f"trips over 24 h           : {int((df['ride_length_min'] > 24 * 60).sum()):,}")
print(f"zero-distance round trips : {int((df['trip_distance_km'] == 0).sum()):,}")
print(f"trips over 4000 km        : {int((df['trip_distance_km'] > 4000).sum()):,}")
pd.concat([too_long, zero_dist, too_far]).round({"ride_length_min": 0, "trip_distance_km": 2})

#### Different Structure

Citi Bike's trip-file schema changed around **2020**. Files from **2013-2019** (and the Jersey
City `JC-*` files) use a **legacy** layout: space-separated column names, a `bikeid`, a `usertype`
of `Subscriber`/`Customer`, and rider demographics (`gender`, `birth year`); they carry no
`ride_id` and no `rideable_type` (electric bikes did not exist yet). From **2020** onward the
files use the **modern** schema analysed above.


In [ ]:
# Compare the two actual schemas: the modern layout (2020+, loaded above) and the legacy
# pre-2020 layout that `load_rides` maps onto it via LEGACY_RENAME.
MODERN = ["ride_id", "rideable_type", "started_at", "ended_at",
          "start_station_name", "start_station_id", "end_station_name", "end_station_id",
          "start_lat", "start_lng", "end_lat", "end_lng", "member_casual"]
assert all(c in df.columns for c in MODERN)  # these are the loaded modern file's raw columns

modern_to_legacy = {v: k for k, v in LEGACY_RENAME.items()}
modern_to_legacy["member_casual"] = "usertype"
notes = {
    "ride_id": "added - unique trip id",
    "rideable_type": "added - classic vs electric (e-bikes)",
    "member_casual": "renamed + recoded (Subscriber->member, Customer->casual)",
}
rows = [{"Modern (2020+)": c,
         "Legacy (2013-2019)": modern_to_legacy.get(c, "-"),
         "Change": notes.get(c, "renamed")}
        for c in MODERN]
for leg, note in [("tripduration", "dropped - derivable from timestamps"),
                  ("bikeid", "dropped - physical bike id"),
                  ("birth year", "dropped - rider demographic"),
                  ("gender", "dropped - rider demographic")]:
    rows.append({"Modern (2020+)": "-", "Legacy (2013-2019)": leg, "Change": note})

print(f"Loaded month uses the {'LEGACY' if meta['is_legacy'] else 'MODERN'} schema.")
pd.DataFrame(rows)

## 2. Station Metadata (GBFS)

Current station information and live availability from the Lyft GBFS feed (the same feed the
backend uses). Two endpoints are merged based on `station_id`: `station_information` (static: name, location, capacity)
and `station_status` (live: bikes/docks available, operational flags).

In [ ]:
info = requests.get(GBFS_INFO_URL, timeout=(5, 30)).json()["data"]["stations"]
status = requests.get(GBFS_STATUS_URL, timeout=(5, 30)).json()["data"]["stations"]
status_map = {s["station_id"]: s for s in status}

rows = []
for s in info:
    st = status_map.get(s["station_id"], {})
    counts = {v.get("vehicle_type_id"): v.get("count", 0)
              for v in st.get("vehicle_types_available", [])}
    rows.append({
        "station_id": s["station_id"],
        "short_name": s.get("short_name"),
        "name": s.get("name"),
        "lat": s.get("lat"),
        "lon": s.get("lon"),
        "capacity": s.get("capacity"),
        "num_bikes_available": st.get("num_bikes_available"),
        "num_classic": counts.get("1"),
        "num_ebikes": counts.get("2"),
        "num_ebikes_reported": st.get("num_ebikes_available"),
        "num_docks_available": st.get("num_docks_available"),
        "num_bikes_disabled": st.get("num_bikes_disabled"),
        "num_docks_disabled": st.get("num_docks_disabled"),
        "is_installed": st.get("is_installed"),
        "is_renting": st.get("is_renting"),
        "is_returning": st.get("is_returning"),
    })
stations = pd.DataFrame(rows)
stations["active"] = (
    (stations["is_installed"] == 1) & (stations["is_renting"] == 1) & (stations["is_returning"] == 1)
)
print(f"Stations in feed: {len(stations)}  (active: {stations['active'].sum()})")

# The feed lists not-installed stations first, so a plain head() would show only those.
preview = pd.concat([stations[stations["active"]].head(3),
                     stations[~stations["active"]].head(2)])
preview

### 2.1 Structure

We provide a brief overview of the merged table's structure, including column names, data types, and sample records, through the `pandas` library.

Two **extracted features** are derived on top of the raw feed fields. `active` combines the three operational flags (`is_installed`, `is_renting`, `is_returning`) into a single boolean, used to filter unavailable stations out of live views. `num_classic` and `num_ebikes` split the bikes at a station by vehicle type: the feed reports availability as a list of `(vehicle_type_id, count)` pairs, so it is indexed by type id (`"1"` classic, `"2"` electric), and a type absent from that list leaves no count.

In [ ]:
stations.info()

print("\nExtracted features (active and inactive stations):")
print(preview[["short_name", "num_bikes_available", "num_classic", "num_ebikes",
               "is_installed", "is_renting", "is_returning", "active"]])

### 2.2 Station Capacity

We look at how the network is provisioned, through the **distribution of station capacity**: how many docks each station has, and how many stations share a given size.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(stations["capacity"].dropna(), bins=30)
plt.title("Station capacity")
plt.xlabel("Docks per station")
plt.ylabel("Number of stations")
plt.tight_layout()
plt.show()

print(f"Stations                  : {len(stations)}")
print(f"Median docks per station  : {stations['capacity'].median():.0f}")
print(f"Stations with capacity 0  : {int((stations['capacity'] == 0).sum())}")

Capacity distinguishes **small stations** from the **larger hubs** that anchor the busiest areas, which is what the dashboard encodes when sizing station markers. The bar at **0 docks** is not corrupt data: those are stations that appear in the feed but are not physically installed.

### 2.3 Live Availability

We now turn to the current state of the network. Rather than a few headline totals, the chart decomposes the network's **entire capacity**, so every dock the feed knows about is counted exactly once, in one of six states: holding a rentable **classic bike** or **e-bike**, **free** to receive one, holding a bike that is **out of service**, being a **dock** that is itself out of service, or **unreported**, meaning capacity that no counter accounts for. The same breakdown is repeated for active and inactive stations, whose composition turns out to be entirely different.

In [ ]:
# Live snapshot: these totals change on every run.
def dock_breakdown(d):
    b = pd.Series({
        "Classic bikes": d["num_classic"].sum(),
        "E-bikes": d["num_ebikes"].sum(),
        "Free docks": d["num_docks_available"].sum(),
        "Bikes out of service": d["num_bikes_disabled"].sum(),
        "Docks out of service": d["num_docks_disabled"].sum(),
    })
    b["Unreported"] = d["capacity"].sum() - b.sum()  # capacity no counter accounts for
    return b

groups = {
    "Whole network": stations,
    "Active stations": stations[stations["active"]],
    "Inactive stations": stations[~stations["active"]],
}
parts = pd.DataFrame({k: dock_breakdown(d) for k, d in groups.items()}).T
share = 100 * parts.div(parts.sum(axis=1), axis=0)

colors = dict(zip(share.columns,
                  ["#2a7f62", "#7fc4a8", "#4c78a8", "#e07b39", "#b0561a", "#bdbdbd"]))
STEP = 1.7  # vertical spacing between bars, leaving room for the call-outs around each
fig, ax = plt.subplots(figsize=(10, 5))
for i, g in [(n * STEP, g) for n, g in enumerate(share.index)]:
    left, thin = 0.0, 0
    for col in share.columns:
        s, n = share.loc[g, col], parts.loc[g, col]
        ax.barh(i, s, left=left, color=colors[col], height=0.6,
                label=col if i == 0 else None)
        if s >= 4:   # wide enough to hold the value inside the segment
            ax.text(left + s / 2, i, f"{s:.0f}%\n{int(n):,}", ha="center", va="center",
                    fontsize=9, color="white")
        elif n > 0:  # too thin to write in: call it out beside the bar
            x = left + s / 2
            up = thin % 2 == 0  # alternate above / below so leaders never cross a label
            ax.annotate(f"{col}: {s:.1f}% ({int(n):,})",
                        xy=(x, i - 0.3 if up else i + 0.3),
                        xytext=(x, i - 0.42 if up else i + 0.42),
                        ha="right" if x > 80 else "left" if x < 20 else "center",
                        va="bottom" if up else "top", fontsize=8, color="0.25",
                        clip_on=False, arrowprops=dict(arrowstyle="-", lw=0.7, color="0.5"))
            thin += 1
        left += s

ax.set_yticks([n * STEP for n in range(len(share))])
ax.set_yticklabels(share.index)
ax.set(xlim=(0, 100), ylim=((len(share) - 1) * STEP + 0.95, -0.95),
       xlabel="Share of the group's docks (%)", ylabel="",
       title="Live system-wide availability")
ax.grid(axis="y", visible=False)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.26), frameon=False, fontsize=11)
plt.tight_layout()
plt.show()

summary = parts.assign(**{"Total docks": parts.sum(axis=1)})
summary.insert(0, "Stations", [len(d) for d in groups.values()])
print(summary.astype(int).to_string())

n_inactive = len(groups["Inactive stations"])
inactive_docks = int(parts.loc["Inactive stations"].sum())
print(f"\nInactive stations              : {n_inactive:,} of {len(stations):,} "
      f"({100 * n_inactive / len(stations):.1f}%), holding {inactive_docks:,} docks "
      f"({100 * inactive_docks / parts.loc['Whole network'].sum():.1f}% of capacity)")

bikes = stations["num_bikes_available"].sum()
print(f"E-bikes among available bikes  : "
      f"{100 * stations['num_ebikes'].sum() / bikes:.0f}% of {int(bikes):,}")
print(f"Unreported docks               : {int(parts.loc['Whole network', 'Unreported']):,} "
      f"network-wide, of which {int(parts.loc['Inactive stations', 'Unreported']):,} "
      f"at inactive stations")
print(f"Bikes left at inactive stations: "
      f"{int(parts.loc['Inactive stations', ['Classic bikes', 'E-bikes']].sum()):,} rentable, "
      f"{int(parts.loc['Inactive stations', 'Bikes out of service']):,} out of service")

Approximately half of system docks hold rentable bikes, with e-bikes constituting a growing majority. With the network operating above 50% capacity, available bikes outnumber open docks. Out-of-service bikes occupy a small subset of docks without generating supply, whereas out-of-service docks remain rare.

The remaining capacity represents an "unreported" reconciliation gap (Section 2.4). Categorizing by station status shows that active stations balance cleanly, while inactive stations declare capacity while returning zero counts. This discrepancy is an artifact of the feed's registry structure rather than an operational miscount. Furthermore, inactive stations occasionally retain out-of-service bikes docked before removal from service.

### 2.4 Data Quality

#### Unreported Capacity

The grey slice in section 2.3 is the share of the network's docks that **no live counter accounts for**, and it is the largest quality issue in this dataset.
The plot below resolves the gap per station, in order to better understand its distribution.

In [ ]:
# The dock identity: bikes available + free docks + bikes out of service + docks out of
# service should equal capacity. Whatever is left over is capacity that no counter reports.
counted = (stations["num_bikes_available"] + stations["num_docks_available"]
           + stations["num_bikes_disabled"] + stations["num_docks_disabled"])
unreported = stations["capacity"] - counted  # > 0 docks nobody reports, < 0 counters above capacity
act, inact = stations["active"], ~stations["active"]
short = unreported > 0

print(f"Unreported docks (capacity - counters) : {int(unreported[short].sum()):,}")
print(f"  at inactive stations : {int(unreported[inact & short].sum()):,} "
      f"over {int((inact & short).sum())} stations")
print(f"  at active stations   : {int(unreported[act & short].sum()):,} "
      f"over {int((act & short).sum())} stations "
      f"(median {unreported[act & short].median():.0f}, max {int(unreported[act & short].max())} docks)")
print(f"  counters > capacity  : {int(-unreported[unreported < 0].sum()):,} docks "
      f"over {int((unreported < 0).sum())} stations")

# Per-station distribution, over the stations that declare any dock at all.
sized = stations["capacity"] > 0
print(f"\nStations with capacity > 0 : {int(sized.sum()):,}")
print(f"  reconcile exactly       : {int((unreported[sized] == 0).sum()):,} "
      f"({100 * (unreported[sized] == 0).mean():.1f}%)")
print(f"  within +/- 2 docks      : {100 * (unreported[sized].abs() <= 2).mean():.1f}%")
print(f"  largest single gap      : {int(unreported[sized].max())} docks unreported")

plt.figure(figsize=(8, 4))
sns.histplot(unreported[sized].clip(lower=-5, upper=15), bins=21)
plt.title("Unreported capacity per station")
plt.xlabel("Capacity - counted docks  (0 = consistent; tails folded at -5 / +15)")
plt.ylabel("Number of stations")
plt.tight_layout()
plt.show()

#### Identifier Consistency

The feed names a station **twice**, and the two identifiers are not interchangeable: `station_id` is internal to GBFS, while `short_name` is the public code of the station. Which of the two the trip files carry decides whether the live metadata can be joined to the ride history at all, so the choice is **verified below rather than assumed**.

In [ ]:
# Which identifier joins the two datasets? The trips carry only one of the feed's two.
trip_ids = pd.concat([df["start_station_id"], df["end_station_id"]]).dropna().astype(str)
distinct = set(trip_ids.unique())
short_names = set(stations["short_name"].astype(str))
gbfs_ids = set(stations["station_id"].astype(str))

print(f"Distinct station ids in the trips : {len(distinct):,}")
print(f"  matching short_name : {len(distinct & short_names):,} "
      f"({100 * len(distinct & short_names) / len(distinct):.1f}%)")
print(f"  matching station_id : {len(distinct & gbfs_ids):,}")

# station_id is not even uniform in shape, which is part of why it is unusable as a public key.
is_uuid = stations["station_id"].str.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-.*")
print(f"\nstation_id shaped as a UUID : {int(is_uuid.sum()):,} / {len(stations):,} "
      f"(the rest are numeric strings)")

# What the unmatched trip ids are made of.
unmatched = distinct - short_names
truncated = {i for i in unmatched if re.fullmatch(r"\d+\.\d", i) and f"{i}0" in short_names}
non_station = {i for i in unmatched if not re.fullmatch(r"[\d.]+", i)}
print(f"\nTrip ids absent from the feed : {len(unmatched):,}")
print(f"  truncated decimals, recovered by padding : {len(truncated)}")
print(f"  depots, system and demo entries          : {len(non_station)} "
      f"{sorted(non_station)[:4]}")
print(f"  retired stations                         : "
      f"{len(unmatched) - len(truncated) - len(non_station)}")

legs = int(trip_ids.isin(truncated).sum())
print(f"\nStation legs lost to the truncated ids : {legs:,} "
      f"({100 * legs / len(trip_ids):.2f}% of {len(trip_ids):,})")

# The station name cannot serve as a key either.
dup_names = stations["name"].value_counts()
dup_names = dup_names[dup_names > 1]
print(f"Names shared by more than one station  : {len(dup_names)} "
      f"{list(dup_names.index)}")

#### Missing Values

The table below is the proof that no field we consume is missing. Nulls alone would be weak evidence, since a missing value can hide as a sentinel, so every column is also checked for values that are present but cannot be real: a coordinate at the zero island or outside New York, a blank identifier, a negative count.

In [ ]:
NYC_LAT_RANGE, NYC_LON_RANGE = (40.4, 41.1), (-74.3, -73.6)

def suspect(col):
    """Values that are present but cannot be real."""
    s = stations[col]
    if col in ("lat", "lon"):
        lo, hi = NYC_LAT_RANGE if col == "lat" else NYC_LON_RANGE
        return int(((s == 0) | (s < lo) | (s > hi)).sum())   # zero island / outside NYC
    if s.dtype == bool:
        return 0
    if s.dtype.kind in "if":
        return int((s < 0).sum())                            # counts cannot be negative
    return int((s.astype(str).str.strip() == "").sum())      # blank text

completeness = pd.DataFrame({
    "dtype": stations.dtypes.astype(str),
    "nulls": stations.isna().sum(),
    "null %": (stations.isna().mean() * 100).round(2),
    "suspect values": [suspect(c) for c in stations.columns],
})
print(f"{len(stations):,} stations; per-column completeness:")
completeness

## 3. Weather Data

Hourly NYC weather from the [Open-Meteo archive API](https://open-meteo.com/), used to relate
ridership to weather.

In [ ]:
ride_year = df["ended_at"].min().year
start_date = date(ride_year, 1, 1)
end_date = min(date(ride_year, 12, 31), date.today() - timedelta(days=1))

# Ask for UTC and localise here rather than passing timezone=America/New_York
resp = requests.get(
    WEATHER_API_URL,
    params={
        "latitude": NYC_LAT,
        "longitude": NYC_LON,
        "start_date": (start_date - timedelta(days=1)).isoformat(),
        "end_date": (end_date + timedelta(days=1)).isoformat(),
        "hourly": "temperature_2m,precipitation,weather_code,wind_speed_10m",
        "timezone": "UTC",
        "wind_speed_unit": "kmh",
    },
    timeout=(5, 120),
)
resp.raise_for_status()

weather = pd.DataFrame(resp.json()["hourly"])
weather["datetime"] = (pd.to_datetime(weather["time"], utc=True)
                       .dt.tz_convert(NYC_TZ).dt.tz_localize(None))
weather = (weather[weather["datetime"].dt.date.between(start_date, end_date)]
           .drop(columns="time").reset_index(drop=True))
print(f"Fetched {len(weather):,} hourly rows: {start_date} -> {end_date}")
weather.head()

### 3.1 Structure and Summary

In [ ]:
weather.info()
print("\nSummary statistics:")
weather[["temperature_2m", "wind_speed_10m", "precipitation", "weather_code"]].describe()

### 3.2 Ridership vs. Weather

We join the hourly weather data with the trip data to analyze how weather conditions affect ridership patterns. The analysis focuses on key weather variables such as temperature and precipitation, and their correlation with the number of trips taken.

This helps identify how environmental factors influence bike usage, providing insights for operational planning and demand forecasting.

In [ ]:
# Join hourly ride counts to the matching weather hour.
df["hour_ts"] = df["ended_at"].dt.floor("h")
rides_per_hour = df.groupby("hour_ts").size().reset_index(name="rides")
merged = rides_per_hour.merge(weather, left_on="hour_ts", right_on="datetime", how="inner")
print(f"{len(merged):,} hours joined "
      f"({merged['hour_ts'].min():%Y-%m-%d} -> {merged['hour_ts'].max():%Y-%m-%d})")

# Daily granularity
daily = merged.set_index("hour_ts").resample("D").agg(
    rides=("rides", "sum"),
    temperature=("temperature_2m", "mean"),
    precipitation=("precipitation", "sum"))

# Small multiples on a shared date axis
RIDES_C, TEMP_C, PRECIP_C = "#2a78d6", "#eb6834", "#1baf7a"
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(daily.index, daily["rides"], color=RIDES_C, lw=2, marker="o", ms=4)
axes[0].set_ylabel("Rides per day", color=RIDES_C)
axes[0].set_title("Daily ridership")

axes[1].plot(daily.index, daily["temperature"], color=TEMP_C, lw=2, marker="o", ms=4)
axes[1].set_ylabel("Mean temperature (°C)", color=TEMP_C)
axes[1].set_title("Daily mean temperature")
axes[1].axhline(0, color="0.6", lw=1, ls=":", zorder=0)

# Precipitation is a daily accumulation that is zero on most days
axes[2].bar(daily.index, daily["precipitation"], color=PRECIP_C, width=0.7)
axes[2].set_ylabel("Precipitation (mm/day)", color=PRECIP_C)
axes[2].set_title("Daily precipitation")

# The report claims the least-ridden day is also the wettest, so check it rather than assume it.
worst = daily["rides"].idxmin()
wettest = daily["precipitation"].idxmax()
for ax in axes:
    ax.axvline(worst, color="0.35", lw=1.2, ls="--", zorder=0)
axes[0].annotate(f"{worst:%b %d}: {daily.loc[worst, 'precipitation']:.0f} mm rain"
                 f"{' (month peak)' if worst == wettest else ''},\n"
                 f"lowest ridership of the month",
                 xy=(worst, daily.loc[worst, "rides"]), xytext=(58, 26),
                 textcoords="offset points", ha="center", fontsize=8.5,
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7", alpha=0.9),
                 arrowprops=dict(arrowstyle="->", color="black", lw=1))

axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[2].set_xlabel("Day")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"least-ridden day = {worst:%b %d}, wettest day = {wettest:%b %d} "
      f"-> {'the same day' if worst == wettest else 'different days'}")
print(f"corr(rides, temperature)   = {daily['rides'].corr(daily['temperature']):+.3f}")
print(f"corr(rides, precipitation) = {daily['rides'].corr(daily['precipitation']):+.3f}")
# Same weekday, one wet and one dry, to separate rain from the weekend effect.
print(daily.loc[[worst, worst + pd.Timedelta(days=7)]].round(1))

This justifies weather as an explanatory variable for demand, which is why this dataset is
part of the project.

### 3.3 Data Quality

The feed is a well known and properly maintained source, so the checks here are limited to
completeness and to the one handling detail that is easy to get wrong: the **timezone**. The
archive applies a single fixed UTC offset to a whole request, whichever one New York happens to
be on the day the request runs, so asking it for local time silently mislabels every date on the
far side of a DST boundary by an hour. We therefore request UTC and convert ourselves (section 3
above), which is why the expected row count below is DST-aware rather than a flat 24 hours per
day: the spring-forward day has 23 hours and the autumn fall-back day has 25.

The table reports the missing count and percentage for every raw column.

In [ ]:
raw_cols = ["datetime", "temperature_2m", "precipitation", "weather_code", "wind_speed_10m"]
missing = weather[raw_cols].isna().sum()
missing = (pd.DataFrame({"missing": missing, "pct (%)": (missing / len(weather) * 100).round(2)})
           .sort_values("missing", ascending=False))
# Nulls alone would not catch an absent row, so check the hour count as well. Counting local
# hours rather than days * 24 keeps this honest across the DST transitions.
expected_hours = len(pd.date_range(start_date, end_date + timedelta(days=1),
                                   freq="h", tz=NYC_TZ, inclusive="left"))
print(f"{len(weather):,} hourly rows loaded, {expected_hours:,} expected "
      f"({start_date} -> {end_date}); missing values per column:")
missing

## 4. Bike Lanes

NYC's bike-route network from [NYC OpenData](https://data.cityofnewyork.us/) (dataset `mzxg-pwib`),
used to render bike infrastructure. Each row is a route segment with a `the_geom` geometry, the
street, facility class, borough, status and installation/retirement dates.

In [ ]:
routes = pd.read_csv(BIKE_ROUTES_URL, low_memory=False)
print("Shape:", routes.shape)
print("Columns:", routes.columns.tolist())
routes.head()

### 4.1 Breakdowns

In [ ]:
BORO = {1: "Manhattan", 2: "Bronx", 3: "Brooklyn", 4: "Queens", 5: "Staten Island"}
routes["boro_name"] = routes["boro"].map(BORO)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
routes["boro_name"].value_counts().plot.bar(ax=axes[0], title="Segments by borough")
routes["facilitycl"].value_counts().plot.bar(ax=axes[1], title="Segments by facility class")
routes["status"].value_counts().plot.bar(ax=axes[2], title="Segments by status")
for ax in axes:
    ax.set_ylabel("Segments")
plt.tight_layout()
plt.show()

Segments concentrate in Manhattan and Brooklyn, and most are **current** rather than retired. The
facility class encodes the level of protection (e.g. protected paths vs. shared/sharrow lanes),
which matters for how the network is rendered in the visualization.

### 4.2 Installation Timeline

In [ ]:
routes["install_year"] = pd.to_datetime(
    routes["instdate"], errors="coerce"
).dt.year
by_year = routes["install_year"].value_counts().sort_index()

plt.figure(figsize=(12, 4))
sns.barplot(x=by_year.index.astype(int), y=by_year.values)
plt.xticks(rotation=45)
plt.xlabel("Installation year")
plt.ylabel("Segments installed")
plt.title("Bike-route segments installed per year")
plt.tight_layout()
plt.show()

The installation timeline tracks how NYC's bike network has expanded over the years (note that
`instdate` is missing for some legacy segments, so the earliest years are under-counted).

### 4.3 Map of the Network

In [ ]:
# Extract polylines ([[lat, lon], ...]) from a (MULTI)LINESTRING WKT string.
def parse_wkt_lines(wkt):
    if not isinstance(wkt, str):
        return []
    lines = []
    for group in re.findall(r"\(([^()]+)\)", wkt):
        pts = []
        for pair in group.split(","):
            parts = pair.split()
            if len(parts) >= 2:
                lon, lat = float(parts[0]), float(parts[1])  # WKT is lon-lat
                pts.append([lat, lon])
        if pts:
            lines.append(pts)
    return lines

# Draw a sample of segments to keep the map light.
sample = routes.dropna(subset=["the_geom"]).head(800)
routes_map = folium.Map(location=[NYC_LAT, NYC_LON], zoom_start=11)
for geom in sample["the_geom"]:
    for line in parse_wkt_lines(geom):
        folium.PolyLine(line, color="crimson", weight=2, opacity=0.6).add_to(routes_map)
routes_map

### 4.4 Data Quality

In [ ]:
quality_cols = ["the_geom", "instdate", "ret_date", "facilitycl", "boro", "street"]
miss = routes[quality_cols].isna().sum()
print("Missing values (selected columns):")
print(pd.DataFrame({"missing": miss, "pct": (miss / len(routes) * 100).round(1)}))
print(f"\nUnmapped borough codes: {routes['boro_name'].isna().sum()}")
print(f"Segments with unparseable installation date: {routes['install_year'].isna().sum()}")